Naïve market benchmark (annual S&P 500 returns by GICS sector).
To provide an interpretable baseline against which the proposed ESG disclosure scores can later be evaluated, we first construct a simple benchmark of annual equity performance for S&P 500 constituents, aggregated by Global Industry Classification Standard (GICS) sector. This benchmark is intentionally naïve: it is not intended to identify causal effects, but to supply descriptive “ground truth” variation in returns across time and across sectors that can be compared alongside ESG intensity measures.

This baseline is motivated by the historical role of ESG scoring as an operational proxy for financially material non-financial risks and opportunities, originally framed as inputs to expected future performance. Consistent with this tradition, the subsequent analysis evaluates whether variation in disclosure-based ESG intensity is associated with realised returns in the same annual frequency.

Returns data source.
Security-level price and returns data are retrieved programmatically via the Polygon (Massive.com) Stocks Developer API. Use of this source is appropriate for two reasons: it provides consistent, machine-readable historical market data over the study window, and access is authorised through an API key supplied by the client, ensuring the workflow is reproducible and aligned with the project’s data governance constraints.

In [5]:
import os
import pandas as pd
import yfinance as yf
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------------
# 1. Configuration
# ------------------------------

TICKERS = [
    "SPY",
    "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XLRE"
]

START_DATE = "2014-12-31"   # Pull slightly earlier for 2015 first trading day
END_DATE   = "2024-12-31"

# ------------------------------
# 2. Fetch Adjusted Close Prices
# ------------------------------

print("Downloading data from Yahoo Finance...")

data = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    progress=False,
    auto_adjust=False
)

adj_close = data["Adj Close"].dropna(how="all")

if adj_close.empty:
    raise RuntimeError("No data downloaded. Check internet connection or ticker list.")

# ------------------------------
# 3. Compute Annual Returns
# ------------------------------

returns = []

for ticker in adj_close.columns:
    series = adj_close[ticker].dropna()
    first_prices = series.groupby(series.index.year).first()
    last_prices = series.groupby(series.index.year).last()

    for year in range(2015, 2025):
        if year in first_prices.index and year in last_prices.index:
            annual_return = (last_prices.loc[year] / first_prices.loc[year] - 1) * 100
            returns.append({
                "year": year,
                "ticker": ticker,
                "return_pct": annual_return
            })

df_ret = pd.DataFrame(returns)

if df_ret.empty:
    raise RuntimeError("Annual return calculation failed.")

# ------------------------------
# 4. Plot
# ------------------------------

df_plot = df_ret.sort_values(["ticker", "year"])

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=df_plot,
    x="year",
    y="return_pct",
    hue="ticker",
    marker="o",
    linewidth=2,
    ax=ax
)

ax.axhline(0, color="black", linewidth=1)

ax.set_xlabel("Year")
ax.set_ylabel("Annual Total Return (%)")
ax.set_title("Annual Returns (Adj Close) — SPY & Sector SPDRs, 2015–2024")

# ------------------------------
# 5. Legend Shorthand Relabel
# ------------------------------

label_map = {
    "SPY":  "SPY (Market)",
    "XLB":  "XLB (Mat)",
    "XLC":  "XLC (Comm)",
    "XLE":  "XLE (Engy)",
    "XLF":  "XLF (Fin)",
    "XLI":  "XLI (Ind)",
    "XLK":  "XLK (IT)",
    "XLP":  "XLP (Stap)",
    "XLU":  "XLU (Util)",
    "XLV":  "XLV (HC)",
    "XLY":  "XLY (Disc)",
    "XLRE": "XLRE (RE)",
}

handles, labels = ax.get_legend_handles_labels()

if labels and labels[0] == "ticker":
    handles = handles[1:]
    labels = labels[1:]

new_labels = [label_map.get(l, l) for l in labels]

ax.legend(
    handles=handles,
    labels=new_labels,
    title="ETF (Sector)",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

print("Done.")


ModuleNotFoundError: No module named 'yfinance'

The 2022 drawdown highlights a sectoral divergence, with Energy significantly outperforming the broader index amid inflationary and geopolitical shocks. This regime effect is particularly relevant when interpreting ESG-related return differentials, as fossil fuel exposure provided a temporary relative hedge against tightening financial conditions

Earliest accessible date for SPY: 2016-02-14


In [ ]:
# Install yfinance if not already installed
import sys
!{sys.executable} -m pip install yfinance --quiet
